<a href="https://colab.research.google.com/github/bzeot/colab_inclass/blob/main/hocsaubuoi4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

mô hình tuyến tính không thể xử lý được các mối quan hệ phi tuyến phức tạp nơi dữ liệu biến thiên theo đường cong hoặc hình sóng, đồng thời nó không tự nắm bắt được sự tương tác đa chiều giữa các yếu tố và rất dễ bị sai lệch bởi các giá trị bất thường

In [1]:
dataset_inputs = [
 [1, 6, 7],
 [2, 7, 8],
 [3, 6, 9],
 [4, 8, 9],
 [5, 7, 10],
 [6, 8, 10]
]
true_scores = [6, 8, 9, 12, 14, 16]

In [2]:
weights_input_hidden = [
[0.1, 0.1, 0.1],
[0.2, 0.2, 0.2]
]
print(weights_input_hidden)
print(len(weights_input_hidden))
print(len(weights_input_hidden[0]))

[[0.1, 0.1, 0.1], [0.2, 0.2, 0.2]]
2
3


In [3]:
weights_hidden_output = [0.1, 0.1]

có 2 trọng số vì:Đầu vào: có 2 neuron ở tầng ẩn (đã tạo ở bài trước).Đầu ra: chỉ có 1 neuron ở tầng đầu ra.Kết nối: Mỗi neuron ẩn nối với neuron đầu ra bằng 1 sợi dây (trọng số).Kết luận: 2 neuron ẩn X 1 neuron đầu ra = 2 trọng số.


In [5]:
def elementwise_multiply(list_a, list_b):
  result = []
  for i in range(len(list_a)):
    result.append(list_a[i] * list_b[i])
  return result

In [7]:
def vector_sum(vector):
  total = 0
  for v in vector:
    total += v
  return total

In [8]:
def dot_product(inputs, weights):
  return vector_sum(elementwise_multiply(inputs, weights))

In [23]:
sample = dataset_inputs[0]
print(dot_product(sample,weights_input_hidden[0]))

1.4000000000000001


In [28]:
def hidden_layer_forward(inputs, weights_input_hidden):
  hidden_output = []
  for n in range (len(weights_hidden_output)):
    output = dot_product(inputs, weights_input_hidden[n])
    hidden_output.append(output)
  return hidden_output

In [29]:
hidden_outputs = hidden_layer_forward(sample, weights_input_hidden)
print(hidden_outputs)

[1.4000000000000001, 2.8000000000000003]


In [30]:
def output_layer_forward(hidden_outputs, weights_hidden_output):
  prediction = dot_product(hidden_outputs, weights_hidden_output)
  return prediction

In [31]:
def neural_network(inputs, w_in_hid, w_hid_out):
  hidden_outputs = hidden_layer_forward(inputs, w_in_hid)
  prediction = output_layer_forward(hidden_outputs, w_hid_out)
  return prediction

In [35]:
print(neural_network(sample, weights_input_hidden, weights_hidden_output))

0.42000000000000004


mô hình ở Bài 3 chỉ là một phép tính tuyến tính trực tiếp , thì mô hình này có thêm tầng ẩn để trích xuất các đặc điểm phức tạp hơn. Việc có tầng ẩn giúp biến đổi dữ liệu thô thành những đại diện mới trước khi dự đoán, cho phép mạng thần kinh học được các quy luật phi tuyến tính mà mô hình đơn giản ở Bài 3 không thể làm được

In [36]:
for i in range (len(dataset_inputs)):
  sample = dataset_inputs[i]
  truth = true_scores[i]
  prediction = neural_network(sample, weights_input_hidden, weights_hidden_output)
  error = prediction - truth
  print(error)

-5.58
-7.49
-8.459999999999999
-11.37
-13.34
-15.28


In [39]:
alpha = 0.01
for j in range (len(weights_hidden_output)):
  weights_hidden_output[j] = weights_hidden_output[j] - alpha * error * hidden_outputs[j]

In [41]:
sample = dataset_inputs[0]
truth = true_scores[0]
prediction = neural_network(sample, weights_input_hidden, weights_hidden_output)
error = prediction - truth
hidden_errors = []
for j in range(len(weights_hidden_output)):
    hidden_error = error * weights_hidden_output[j]
    hidden_errors.append(hidden_error)
print(hidden_errors)

[-1.3645297408, -2.4705474815999997]


In [42]:
alpha = 0.01
sample = dataset_inputs[0]
for h in range(len(weights_input_hidden)):
  for i in range(len(weights_input_hidden[h])):
    weights_input_hidden[h][i] -= alpha * hidden_errors[h] * sample[i]
print(weights_input_hidden)

[[0.113645297408, 0.181871784448, 0.195517081856], [0.224705474816, 0.34823284889600004, 0.372938323712]]


In [44]:
def train_one_sample(inputs, truth, weights_input_hidden, weights_hidden_output, alpha):
  hidden_outputs = hidden_layer_forward(inputs, weights_input_hidden)
  prediction = output_layer_forward(hidden_outputs, weights_hidden_output)
  error = prediction - truth
  for j in range(len(weights_hidden_output)):
    weights_hidden_output[j] -= alpha * error * hidden_outputs[j]
  hidden_errors = []
  for j in range(len(weights_hidden_output)):
    hidden_error = error * weights_hidden_output[j]
    hidden_errors.append(hidden_error)
  for h in range(len(weights_input_hidden)):
    for i in range(len(weights_input_hidden[h])):
      weights_input_hidden[h][i] -= alpha * hidden_errors[h] * inputs[i]
  return prediction, error

In [60]:
alpha = 0.01
for i in range(len(dataset_inputs)):
  sample = dataset_inputs[i]
  truth = true_scores[i]
  prediction, error = train_one_sample(sample, truth, weights_input_hidden, weights_hidden_output, alpha)
  print(prediction, error)

nan nan
nan nan
nan nan
nan nan
nan nan
nan nan


In [67]:
def compute_mean_loss_2layer(dataset, truths, weights_in_hid, weights_hid_out):
    total_squared_error = 0
    n = len(dataset)
    for i in range(n):
        prediction = neural_network(dataset[i], weights_in_hid, weights_hid_out)
        error = prediction - truths[i]
        total_squared_error += (error ** 2)
    mean_squared_error = total_squared_error / n
    return mean_squared_error

In [69]:
weights_input_hidden = [[0.1, 0.1, 0.1], [0.2, 0.2, 0.2]]
weights_hidden_output = [0.1, 0.1]
alpha = 0.01
epochs = 30
for epoch in range(epochs):
    for i in range(len(dataset_inputs)):
        train_one_sample(dataset_inputs[i], true_scores[i], weights_input_hidden, weights_hidden_output, alpha)
    loss = compute_mean_loss_2layer(dataset_inputs, true_scores, weights_input_hidden, weights_hidden_output)
    print(epoch, loss)

0 3200.6438249968874
1 nan
2 nan
3 nan
4 nan
5 nan
6 nan
7 nan
8 nan
9 nan
10 nan
11 nan
12 nan
13 nan
14 nan
15 nan
16 nan
17 nan
18 nan
19 nan
20 nan
21 nan
22 nan
23 nan
24 nan
25 nan
26 nan
27 nan
28 nan
29 nan


Epoch 0 loss là khoảng 3200, nhưng từ Epoch 1 trở đi loss trở thành nan. Điều này có nghĩa là Loss không giảm mà đang bị bùng nổ do alpha quá lớn

In [70]:
weights_input_hidden = [[0.1, 0.1, 0.1], [0.2, 0.2, 0.2]]
weights_hidden_output = [0.1, 0.1]
alpha = 0.1
epochs = 30
for epoch in range(epochs):
    for i in range(len(dataset_inputs)):
        train_one_sample(dataset_inputs[i], true_scores[i], weights_input_hidden, weights_hidden_output, alpha)
    loss = compute_mean_loss_2layer(dataset_inputs, true_scores, weights_input_hidden, weights_hidden_output)
    print(epoch, loss)

0 inf
1 nan
2 nan
3 nan
4 nan
5 nan
6 nan
7 nan
8 nan
9 nan
10 nan
11 nan
12 nan
13 nan
14 nan
15 nan
16 nan
17 nan
18 nan
19 nan
20 nan
21 nan
22 nan
23 nan
24 nan
25 nan
26 nan
27 nan
28 nan
29 nan


In [71]:
weights_input_hidden = [[0.1, 0.1, 0.1], [0.2, 0.2, 0.2]]
weights_hidden_output = [0.1, 0.1]
alpha = 0.0001
epochs = 30
for epoch in range(epochs):
    for i in range(len(dataset_inputs)):
        train_one_sample(dataset_inputs[i], true_scores[i], weights_input_hidden, weights_hidden_output, alpha)
    loss = compute_mean_loss_2layer(dataset_inputs, true_scores, weights_input_hidden, weights_hidden_output)
    print(epoch, loss)

0 113.45121359669413
1 110.06013987070696
2 106.37614753097647
3 102.36906121315366
4 98.01762658826568
5 93.31180827161013
6 88.25522251775591
7 82.86741501015105
8 77.1856069620962
9 71.26547646240748
10 65.18054861768333
11 59.019867245466514
12 52.883827008678274
13 46.878343086618365
14 41.10787375926225
15 35.66810624805432
16 30.639275837183202
17 26.08104654475372
18 22.029629362383194
19 18.497410328413945
20 15.474914360014617
21 12.934560248199901
22 10.835452222336613
23 9.128429571803169
24 7.760725980200441
25 6.679807930168406
26 5.836195367773534
27 5.185264147286882
28 4.68816198293212
29 4.312035406728282


In [73]:
dataset_inputs = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
true_scores = [10, 20, 30]
weights_input_hidden = [[0.1, 0.2, 0.3], [0.1, 0.2, 0.3], [0.1, 0.2, 0.3]]
weights_hidden_output = [0.1, 0.2, 0.3]

In [74]:
alpha = 0.001
epochs = 30
for epoch in range(epochs):
    for i in range(len(dataset_inputs)):
        train_one_sample(dataset_inputs[i], true_scores[i], weights_input_hidden, weights_hidden_output, alpha)
    loss = compute_mean_loss_2layer(dataset_inputs, true_scores, weights_input_hidden, weights_hidden_output)
    print(epoch, loss)
print(weights_input_hidden)
print(weights_hidden_output)

0 217.05319268069522
1 45.88676319447227
2 3.6564515980797148
3 1.6075601176103307
4 1.5067787768493435
5 1.4925150860693754
6 1.4846482466136697
7 1.477326645842604
8 1.4700800260773297
9 1.4628700785847943
10 1.4556935023990836
11 1.4485498391465266
12 1.4414388633498032
13 1.4343603705585533
14 1.4273141601131032
15 1.4203000337141252
16 1.4133177952768723
17 1.4063672508903229
18 1.3994482087853022
19 1.3925604793038027
20 1.3857038748688908
21 1.3788782099550383
22 1.3720833010590094
23 1.365318966671132
24 1.3585850272471276
25 1.3518813051802505
26 1.3452076247740348
27 1.3385638122153418
28 1.3319496955478962
29 1.3253651046462078
[[0.26716837611681116, 0.43343865404022247, 0.5997089319636333], [0.32107122254747905, 0.50660304297852, 0.692134863409562], [0.3749740689781477, 0.5797674319168189, 0.7845607948554887]]
[0.6351688465478532, 0.7868681682380596, 0.938567489928265]


1. Hidden layer giúp mô hình làm được điều gì?
Giúp mạng học được các đặc trưng phức tạp và xử lý các mối quan hệ phi tuyến tính trong dữ liệu.
2. Backprop thực chất là gì (bằng lời)?
Là quá trình tính toán mức độ đóng góp của từng trọng số vào sai số cuối cùng để điều chỉnh chúng.
3. Vì sao error phải “chảy ngược”?
Vì sai số chỉ xuất hiện ở đầu ra, nên phải truyền ngược về trước mới biết cần sửa các trọng số ở tầng trước như thế nào.
4. So sánh Bài 3 và Bài 4
Bài 3 là mô hình tuyến tính đơn giản chỉ học được đường thẳng. Bài 4 có tầng ẩn nên linh hoạt và giải quyết được vấn đề phức tạp hơn.
5. Khi nào cần nhiều hidden layer hơn?
Khi dữ liệu cực kỳ phức tạp và chứa những quy luật tinh vi mà một tầng ẩn không thể mô tả hết.